# AgriInsight AI
## AI-Powered Crop Yield & Farm Decision Analytics Platform

**Internship:** AICTE | IBM SkillsBuild - Data Analytics with AI Internship 2026  
**Submitted by:** Thodupunuri Sai Charan  
**Program:** B.Tech - CSE (Data Science)

### Goal
Convert historical agricultural data into KPIs, trends, machine-learning predictions and decision-support notes. The notebook follows the internship mindset: **start from data and end with an action/decision**.

## 1. Dataset and Project Scope

**Dataset:** Crop Yield Prediction Dataset  
**Source:** https://www.kaggle.com/datasets/patelris/crop-yield-prediction-dataset

The dataset combines crop yield with rainfall, pesticide-use and average-temperature information. The target is `hg/ha_yield`.

The notebook first checks for a local `yield_df.csv`. If it is not available, it attempts to read a public raw mirror. For offline use, download the CSV from Kaggle and place it in the same folder.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)

LOCAL_DATASET = Path('yield_df.csv')
PUBLIC_MIRROR = 'https://raw.githubusercontent.com/ManikantaSanjay/crop_yield_prediction_regression/master/yield_df.csv'
TARGET = 'hg/ha_yield'
CATEGORICAL = ['Area', 'Item']
NUMERIC = ['Year', 'average_rain_fall_mm_per_year', 'pesticides_tonnes', 'avg_temp']
FEATURES = CATEGORICAL + NUMERIC

## 2. Load the Data

In [ ]:
def load_dataset():
    if LOCAL_DATASET.exists():
        print('Loading local yield_df.csv')
        return pd.read_csv(LOCAL_DATASET)
    print('Local file not found. Attempting public mirror...')
    try:
        return pd.read_csv(PUBLIC_MIRROR)
    except Exception as exc:
        raise RuntimeError(
            'Dataset could not be loaded. Download yield_df.csv from the Kaggle link in README.md '
            'and place it beside this notebook.'
        ) from exc

df_raw = load_dataset()
print('Raw shape:', df_raw.shape)
df_raw.head()

## 3. Data Cleaning

Cleaning is kept explicit so that the evaluator can see exactly what changes are made to the raw data.

In [ ]:
def clean_data(df):
    data = df.copy()
    unnamed = [c for c in data.columns if c.lower().startswith('unnamed')]
    if unnamed:
        data = data.drop(columns=unnamed)

    required = FEATURES + [TARGET]
    missing = [c for c in required if c not in data.columns]
    if missing:
        raise ValueError(f'Missing required columns: {missing}')

    for col in NUMERIC + [TARGET]:
        data[col] = pd.to_numeric(data[col], errors='coerce')

    data[CATEGORICAL] = data[CATEGORICAL].apply(lambda s: s.astype(str).str.strip())
    data = data.replace([np.inf, -np.inf], np.nan)
    data = data.dropna(subset=required)
    data = data.drop_duplicates().reset_index(drop=True)
    data = data[(data[TARGET] > 0) &
                (data['average_rain_fall_mm_per_year'] >= 0) &
                (data['pesticides_tonnes'] >= 0)].copy()
    return data

df = clean_data(df_raw)
print('Clean shape:', df.shape)
print('Missing required values:', int(df[FEATURES + [TARGET]].isna().sum().sum()))
df.head()

## 4. Executive KPIs

These KPIs answer the first BI question: **what is happening?**

In [ ]:
def t_per_ha(hg_per_ha):
    return hg_per_ha / 10000.0

kpis = {
    'Clean records': len(df),
    'Countries / areas': df['Area'].nunique(),
    'Crop types': df['Item'].nunique(),
    'Average yield (t/ha)': round(t_per_ha(df[TARGET].mean()), 2),
    'Median yield (t/ha)': round(t_per_ha(df[TARGET].median()), 2),
}
pd.Series(kpis, name='Value')

## 5. Exploratory Data Analysis

The objective is to identify trends and possible drivers without treating correlation as causation.

In [ ]:
crop_summary = (
    df.groupby('Item', as_index=False)[TARGET].mean()
      .assign(avg_yield_t_ha=lambda x: x[TARGET] / 10000)
      .sort_values('avg_yield_t_ha', ascending=False)
)

plt.figure(figsize=(10, 5))
plt.bar(crop_summary['Item'], crop_summary['avg_yield_t_ha'])
plt.title('Average Yield by Crop')
plt.xlabel('Crop')
plt.ylabel('Average Yield (t/ha)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

crop_summary[['Item', 'avg_yield_t_ha']].head(10)

In [ ]:
year_summary = df.groupby('Year', as_index=False)[TARGET].mean()
year_summary['avg_yield_t_ha'] = year_summary[TARGET] / 10000

plt.figure(figsize=(10, 5))
plt.plot(year_summary['Year'], year_summary['avg_yield_t_ha'], marker='o', linewidth=1.5)
plt.title('Historical Average Yield Trend')
plt.xlabel('Year')
plt.ylabel('Average Yield (t/ha)')
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
numeric_for_corr = NUMERIC + [TARGET]
corr = df[numeric_for_corr].corr(numeric_only=True)[TARGET].drop(TARGET)
print('Pearson correlation with yield (exploratory only):')
display(corr.sort_values(key=np.abs, ascending=False).round(3).to_frame('correlation'))

In [ ]:
sample_crop = df['Item'].value_counts().index[0]
sub = df[df['Item'] == sample_crop]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(sub['average_rain_fall_mm_per_year'], sub[TARGET] / 10000, alpha=0.35)
axes[0].set_title(f'Rainfall vs Yield - {sample_crop}')
axes[0].set_xlabel('Rainfall (mm/year)')
axes[0].set_ylabel('Yield (t/ha)')

axes[1].scatter(sub['avg_temp'], sub[TARGET] / 10000, alpha=0.35)
axes[1].set_title(f'Temperature vs Yield - {sample_crop}')
axes[1].set_xlabel('Average Temperature (°C)')
axes[1].set_ylabel('Yield (t/ha)')

plt.tight_layout()
plt.show()

## 6. Machine-Learning Pipeline

The model uses country/area, crop, year, rainfall, pesticide use and temperature. Linear Regression is used as a baseline. Random Forest is used as the non-linear ensemble model.

In [ ]:
X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

def make_pipeline(model):
    preprocessor = ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL),
        ('num', StandardScaler(), NUMERIC),
    ])
    return Pipeline([('preprocessor', preprocessor), ('model', model)])

models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(
        n_estimators=140,
        max_depth=20,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1,
    ),
}

trained = {}
results = []
for name, model in models.items():
    pipe = make_pipeline(model)
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    results.append({
        'Model': name,
        'MAE_t_ha': mean_absolute_error(y_test, pred) / 10000,
        'RMSE_t_ha': mean_squared_error(y_test, pred) ** 0.5 / 10000,
        'R2': r2_score(y_test, pred),
    })
    trained[name] = pipe

results_df = pd.DataFrame(results).sort_values('R2', ascending=False).reset_index(drop=True)
results_df.round(4)

## 7. Model Reliability Gate

A prediction should not be shown as certain just because the interface looks good. The reliability gate uses held-out test R² to label the model before presenting the prediction.

In [ ]:
best_name = results_df.iloc[0]['Model']
best_model = trained[best_name]
best_r2 = float(results_df.iloc[0]['R2'])

def reliability_label(r2):
    if r2 >= 0.75:
        return 'Strong validation signal'
    if r2 >= 0.40:
        return 'Moderate validation signal'
    if r2 > 0:
        return 'Weak validation signal'
    return 'Low reliability'

print('Selected model:', best_name)
print('Validation status:', reliability_label(best_r2))
print('Test R²:', round(best_r2, 4))

## 8. Example Prediction and Decision-Support Notes

The recommendations below are checks to consider, not fertilizer/pesticide prescriptions.

In [ ]:
example = pd.DataFrame([{
    'Area': df['Area'].mode()[0],
    'Item': df['Item'].mode()[0],
    'Year': int(df['Year'].max()),
    'average_rain_fall_mm_per_year': float(df['average_rain_fall_mm_per_year'].median()),
    'pesticides_tonnes': float(df['pesticides_tonnes'].median()),
    'avg_temp': float(df['avg_temp'].median()),
}])

pred_hg = float(best_model.predict(example)[0])
print('Example input:')
display(example)
print('Predicted yield:', round(pred_hg / 10000, 2), 't/ha')

In [ ]:
def percentile_position(series, value):
    return float((series <= value).mean() * 100)

def decision_notes(df, row, predicted_hg):
    crop = row.iloc[0]['Item']
    crop_df = df[df['Item'] == crop]
    notes = []

    pct = percentile_position(crop_df[TARGET], predicted_hg)
    if pct < 25:
        notes.append('Yield-risk flag: prediction is in the lower historical quartile for this crop.')
    elif pct > 75:
        notes.append('Opportunity flag: prediction is in the upper historical quartile for this crop.')
    else:
        notes.append('Typical-range flag: prediction is within the middle historical range.')

    rain = float(row.iloc[0]['average_rain_fall_mm_per_year'])
    q1, q3 = crop_df['average_rain_fall_mm_per_year'].quantile([0.25, 0.75])
    if rain < q1:
        notes.append('Rainfall is below the crop historical lower quartile: review water availability.')
    elif rain > q3:
        notes.append('Rainfall is above the crop historical upper quartile: review drainage/waterlogging risk.')

    temp = float(row.iloc[0]['avg_temp'])
    tq1, tq3 = crop_df['avg_temp'].quantile([0.25, 0.75])
    if temp < tq1 or temp > tq3:
        notes.append('Temperature is outside the crop central historical range: treat prediction as higher uncertainty.')

    pest = float(row.iloc[0]['pesticides_tonnes'])
    if pest > crop_df['pesticides_tonnes'].quantile(0.75):
        notes.append('Pesticide use is above the crop historical upper quartile: review IPM/local agronomic guidance.')

    return notes

for note in decision_notes(df, example, pred_hg):
    print('-', note)

## 9. Business / Farm Decision View

The project converts analysis into five levels of information:

1. **KPI:** What is happening?
2. **Trend:** Is the value rising, falling or stable?
3. **Driver:** Which variables are associated with the change?
4. **Risk / Opportunity:** What should be reviewed?
5. **Action:** What is the next practical check?

This structure follows the project-discussion guidance that a useful analytics project should end in a meaningful decision, not only in charts.

## 10. IBM SkillsBuild / IBM Bob Integration

IBM Bob is integrated into this project as the **AI-assisted planning, review and project-file workflow** demonstrated during the internship. It is not used as a hidden prediction API. The Python notebook remains the reproducible source of truth for cleaning, EDA, model validation and prediction.

The next cell creates an **IBM Bob Project Brief** from the current cleaned dataset and model results. The brief tells Bob what to review, what not to invent, which files must remain consistent, and that material changes should be proposed for approval before they are accepted.


In [ ]:
def build_ibm_bob_brief(df, results_df, best_name):
    best = results_df.loc[results_df['Model'] == best_name].iloc[0]
    return f'''AGRIINSIGHT AI - IBM BOB PROJECT BRIEF

Project: AgriInsight AI - AI-Powered Crop Yield & Farm Decision Analytics Platform
Internship: AICTE | IBM SkillsBuild - Data Analytics with AI Internship 2026
Student: Thodupunuri Sai Charan

PROBLEM AND OBJECTIVE
Use historical crop, climate and agricultural-input data to produce clear KPIs, EDA, validated yield prediction and evidence-linked decision-support notes. The project must move from raw data to an understandable decision, not only charts or UI.

LIVE DATASET PROFILE
Clean records: {len(df):,}
Countries / areas: {df['Area'].nunique():,}
Crop types: {df['Item'].nunique():,}
Year range: {int(df['Year'].min())}-{int(df['Year'].max())}
Target: {TARGET}
Features: {', '.join(FEATURES)}

CURRENT MODEL VALIDATION
Selected model: {best_name}
MAE: {float(best['MAE_t_ha']):.4f} t/ha
RMSE: {float(best['RMSE_t_ha']):.4f} t/ha
R2: {float(best['R2']):.4f}

FILES TO REVIEW
- AgriInsight_AI.ipynb
- AgriInsight_AI.py
- requirements.txt
- README.md
- AgriInsight_AI_Project_Report.docx
- yield_df.csv or the referenced dataset source

REVIEW TASKS
1. Confirm the problem statement, KPIs and scope match the dataset.
2. Review cleaning, EDA and visualization logic.
3. Review the Linear Regression baseline and Random Forest pipeline.
4. Verify MAE, RMSE and R2 come from the held-out test set.
5. Check requirements.txt, README and project report for consistency.
6. Do not invent soil variables, fertilizer doses or pesticide doses that are absent from the dataset.
7. Keep correlation claims exploratory; do not present them as causation.
8. Preserve the reliability gate and responsible-use notes.

APPROVAL WORKFLOW
First return a short plan and list any proposed changes. Ask for approval before each material change. After approval, provide the revised file(s) and a short explanation.

FINAL DELIVERABLES
Code file, requirements.txt, project report, README and GitHub-ready repository.'''

ibm_bob_brief = build_ibm_bob_brief(df, results_df, best_name)
Path('IBM_Bob_Project_Brief.txt').write_text(ibm_bob_brief, encoding='utf-8')
print(ibm_bob_brief)


## 11. Limitations and Responsible Use

- Historical relationships do not prove causation.
- The dataset is country-level rather than plot-level.
- Soil-test variables are not present and are therefore not invented.
- Model output is decision support, not a guaranteed yield.
- No fertilizer or pesticide dose is prescribed.
- Local agronomic guidance, crop stage, weather and soil testing remain necessary before field action.

## 12. Final Outcome

AgriInsight AI demonstrates the complete Data Analytics with AI workflow: **cleaning, EDA, KPIs, visualization, machine-learning model comparison, model validation, prediction and decision intelligence**. The Streamlit application (`AgriInsight_AI.py`) provides the same workflow as an interactive dashboard.

## IBM Bob repository integration

The GitHub-ready project includes `.bob/skills/agriinsight-review/SKILL.md`, `IBM_Bob_Project_Brief.txt`, and `IBM_Bob_Integration_Guide.md`. These files prepare the plan-first, approval-before-change review workflow demonstrated in the internship. The Python notebook remains the reproducible source of truth for EDA, model validation and predictions.
